In [0]:
spark.sql("DROP TABLE IF EXISTS de_mini_project.silver.store")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS de_mini_project.silver.store (
    store_id STRING,
    location STRING,
    city STRING,
    region STRING,
    contact STRING
)
""")

In [0]:
spark.sql("""
INSERT INTO de_mini_project.silver.store (store_id, location, city, region, contact)
WITH store_cleaned AS (
    SELECT 
        st_id_ AS store_id,
        regexp_replace(location_name_address, '^TechNova - ', '') AS location,
        split(city_region, '_')[0] AS city,
        split(city_region, '_')[1] AS region,
        NULLIF(regexp_replace(regexp_replace(mgr_contact_, 'N/A', ''), '[^0-9]', ''), '') AS contact
    FROM de_mini_project.azure_blob_storage.stores
)
SELECT 
    store_id, 
    location, 
    city, 
    region, 
    contact
FROM store_cleaned
QUALIFY ROW_NUMBER() OVER (PARTITION BY store_id ORDER BY location) = 1
""")

In [0]:
display(spark.sql("SELECT * FROM de_mini_project.silver.store"))